# Multimodal Inconsistency & Fake News Detection (CSC301)

One-click Google Colab (**T4 GPU**) notebook for:
1. Environment & API key setup
2. Streaming the 5,000-pair balanced `g-luo/news-clippings` subset (`3,500 / 750 / 750`)
3. Verifying the frozen `openai/clip-vit-base-patch32` backbone + ~2.46M-parameter Cross-Attention head
4. Training for 5 epochs & saving `checkpoints/best_cross_attention_model.pt`
5. Generating evaluation metrics (`confusion_matrix.png`, `roc_curve.png`, `classification_report.json`)
6. End-to-end `detect_and_verify` demo & fast-vs-escalated latency benchmark

In [ ]:
# 1. Check GPU & Clone / Install Dependencies
!nvidia-smi
import os

if os.path.basename(os.getcwd()) != "Research_ppr":
    if not os.path.exists("Research_ppr"):
        !git clone https://github.com/DevChaudhary596/Research_ppr.git
    %cd Research_ppr
!pip install -q -r requirements.txt

In [ ]:
# 2. Optional API Keys for Live Tier-2 Agent Verification
# If omitted, the agent automatically falls back to DuckDuckGo + extractive summary.
import os

try:
    from google.colab import userdata  # type: ignore
    for key_name in ["TAVILY_API_KEY", "GOOGLE_API_KEY"]:
        try:
            val = userdata.get(key_name)
            if val:
                os.environ[key_name] = val
        except Exception:
            pass
except ImportError:
    pass

print("TAVILY_API_KEY configured:", bool(os.getenv("TAVILY_API_KEY")))
print("GOOGLE_API_KEY configured:", bool(os.getenv("GOOGLE_API_KEY")))

In [ ]:
# 3. Verify Frozen CLIP Backbone & Trainable Parameter Count
from src.models.cross_attention import MultimodalInconsistencyDetector, count_parameters

model = MultimodalInconsistencyDetector()
trainable_p, frozen_p = count_parameters(model)
print(f"Frozen CLIP params:    {frozen_p:,} (~{frozen_p/1e6:.2f}M)")
print(f"Trainable Head params: {trainable_p:,} (~{trainable_p/1e6:.2f}M)")
assert all(not p.requires_grad for p in model.backbone.parameters())

In [ ]:
# 4. Train Cross-Attention Classifier (5 Epochs on T4 GPU, 5,000-pair subset)
from src.training.train import train_model

train_result = train_model(config_path="configs/config.yaml")
print("Best Val Macro-F1:", train_result["best_val_f1"])
print("Checkpoint saved at:", train_result["checkpoint_path"])

In [ ]:
# 5. Evaluate on 750 Test Pairs & Display Seminar Figures
from IPython.display import Image as IPyImage, display
from src.training.evaluate import evaluate_model

eval_metrics = evaluate_model(
    checkpoint_path="checkpoints/best_cross_attention_model.pt",
    assets_dir="assets",
)

display(IPyImage("assets/confusion_matrix.png"))
display(IPyImage("assets/roc_curve.png"))

In [ ]:
# 6. End-to-End Inference Demo & Latency Benchmark
from PIL import Image
from src.pipeline import MultimodalVerificationPipeline, export_ieee_summary

pipeline = MultimodalVerificationPipeline(
    checkpoint_path="checkpoints/best_cross_attention_model.pt"
)

# Demo call (replace with any local image path and news caption)
demo_img = Image.new("RGB", (224, 224), color=(40, 100, 200))
demo_caption = "Crowds gather outside Parliament in London during a 2024 climate summit protest."
result = pipeline.detect_and_verify(demo_img, demo_caption)
print(result)

bench_stats = pipeline.benchmark_latency(num_samples=32, run_agent_sample=True)
print("Latency Benchmark:", bench_stats)

export_ieee_summary(eval_metrics=eval_metrics, benchmark_stats=bench_stats)